In [1]:
import numpy as np
import tensorflow as tf
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
import matplotlib.pyplot as plt
import keras
from sklearn.linear_model import LogisticRegression

In [2]:
data_path = r"C:\Users\Win\Documents\Github Workspace\Machine_Learning\Project\emtab_bone_zscore.csv"
data = pd.read_csv(data_path, delimiter=",")

display(data)

# Separate features and label
X = data.iloc[:, :-1]  # All columns except the last
y = data.iloc[:, -1]   # The last column is assumed to be the label


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

# # Step 2: Split temp into 30% CV and 20% test (from 50% temp = 60/40 split)
X_cv, X_test, y_cv, y_test = train_test_split(
    X_temp, y_temp, test_size=0.65, random_state=42, stratify=y_temp)

# Confirm split sizes
print(f"Train size: {len(X_train)} samples")
print(f"CV size: {len(X_cv)} samples")
print(f"Test size: {len(X_test)} samples")
np.bincount(y)

,201506_at,205486_at,216638_s_at,221619_s_at,221672_s_at,35148_at,Characteristics..Relapse..Metastasis.Bone.
0,0.045283,0.088809,0.359098,0.015206,-0.912406,-1.800676,1
1,-0.875976,-0.041301,-0.111223,1.714917,-0.797470,-1.827605,1
2,0.223085,0.302113,-0.735687,0.131239,-1.416955,2.223924,0
3,0.358691,-0.115612,-0.585816,1.431602,-1.310635,1.830601,1
4,0.789431,0.038531,-1.340381,-0.756025,0.156410,-0.409375,1
...,...,...,...,...,...,...,...
327,-1.140562,-0.295593,-0.806139,-1.139197,-2.230698,-1.492297,0
328,0.940757,-0.189422,0.282271,-0.374587,-1.781049,0.980613,0
329,-0.213690,1.928239,-0.955674,1.031182,-1.498592,0.233792,0
330,0.181532,0.152622,2.391399,-0.289972,-1.949008,-0.260187,0


Train size: 232 samples
CV size: 35 samples
Test size: 65 samples


array([296,  36])

In [ ]:
import pandas as pd
from itertools import combinations
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, recall_score, precision_score,
    f1_score, accuracy_score
)

# Define gene names
gene_names = ['201506_at', '205486_at', '216638_s_at', '221619_s_at', '221672_s_at', '35148_at']

# Convert arrays to DataFrames
X_train = pd.DataFrame(X_train, columns=gene_names)
X_cv = pd.DataFrame(X_cv, columns=gene_names)
X_test = pd.DataFrame(X_test, columns=gene_names)

# Store all results
all_results = []

# Detect classification type
n_classes = len(set(y_train))

# Loop through all feature combinations
for k in range(1, len(gene_names) + 1):
    for comb in combinations(gene_names, k):
        features = list(comb)

        # Select features
        X_train_sel = X_train[features]
        X_cv_sel = X_cv[features]
        X_test_sel = X_test[features]

        # Train Random Forest model
        model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
        model.fit(X_train_sel, y_train)

        # Predict
        y_train_pred = model.predict(X_train_sel)
        y_cv_pred = model.predict(X_cv_sel)
        y_test_pred = model.predict(X_test_sel)
        y_test_proba = model.predict_proba(X_test_sel)

        # AUC handling
        try:
            if n_classes > 2:
                auc = roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro')
            else:
                auc = roc_auc_score(y_test, y_test_proba[:, 1])
        except:
            auc = float('-inf')

        # Other metrics
        recall = recall_score(y_test, y_test_pred, average='macro', zero_division=0)
        precision = precision_score(y_test, y_test_pred, average='macro', zero_division=0)
        f1 = f1_score(y_test, y_test_pred, average='macro', zero_division=0)
        train_acc = accuracy_score(y_train, y_train_pred)
        cv_acc = accuracy_score(y_cv, y_cv_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        all_results.append({
            "genes": ' + '.join(features),
            "auc": auc,
            "recall": recall,
            "precision": precision,
            "f1": f1,
            "train_acc": train_acc,
            "cv_acc": cv_acc,
            "test_acc": test_acc
        })

# Create results DataFrame
results_df = pd.DataFrame(all_results)

# Get best model for each metric
summary_rows = []
metrics = {
    "Highest AUC": "auc",
    "Highest Recall": "recall",
    "Highest Precision": "precision",
    "Highest F1-score": "f1",
    "Highest Train Accuracy": "train_acc",
    "Highest CV Accuracy": "cv_acc",
    "Highest Test Accuracy": "test_acc"
}

for label, metric in metrics.items():
    best_row = results_df.loc[results_df[metric].idxmax()]
    summary_rows.append({
        "Metric": label,
        "Genes": best_row["genes"],
        "AUC": best_row["auc"],
        "Recall": best_row["recall"],
        "Precision": best_row["precision"],
        "F1-score": best_row["f1"],
        "Train Accuracy": best_row["train_acc"],
        "CV Accuracy": best_row["cv_acc"],
        "Test Accuracy": best_row["test_acc"]
    })

# Save summary to CSV
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("Random Forest.csv", index=False)

print("✅ Summary saved to 'best_models_summary_rf.csv'")


✅ Summary saved to 'best_models_summary_rf.csv'
